# Mini Project 1: Analyzing Seattle Crime Patterns by Time, Place, and Offense Type

## Section 1 — Overview

This mini project analyzes the Seattle Police Department Crime Data 2008–Present dataset. The dataset comes from the City of Seattle Open Data portal and includes reported crime incidents, offense dates, report dates, offense categories, offense sub-categories, neighborhoods, precincts, sectors, and beats.

I chose this dataset because reported crime data can show how public safety incidents vary across time and place. From a human-centered design perspective, this matters because civic data can help designers, researchers, and local communities better understand patterns in public information, local services, and urban experiences.

The main questions I investigate are:

1. How do reported crime counts vary by hour of day in Seattle?
2. Which Seattle neighborhoods have the highest number of reported offenses?
3. How do the most common offense sub-categories differ across morning, afternoon, evening, and night?

Together, these questions help me explore temporal patterns, spatial concentration, and offense-type differences in Seattle crime reports.

In [15]:
!pip install jupyter plotly kaleido pandas

In [16]:
import os
import pandas as pd
import plotly.express as px

# Create a folder for saved chart images.
os.makedirs("charts", exist_ok=True)

## Load the Dataset

This notebook uses a CSV file stored inside the `data/` folder so the mini project can be reviewed as a stand-alone artifact. The original Seattle Police Department dataset is large, so this project uses a local CSV copy prepared for this notebook.

In [17]:
# Load the Seattle crime dataset.
df = pd.read_csv("data/seattle_crime_sample.csv")

df.head()

,Report Number,Report DateTime,Offense ID,Offense Date,NIBRS Group AB,NIBRS Crime Against Category,Offense Sub Category,Shooting Type Group,Block Address,Latitude,Longitude,Beat,Precinct,Sector,Neighborhood,Reporting Area,Offense Category,NIBRS Offense Code Description,NIBRS_offense_code,Census Block 2020
0,2016-091229,2016 Mar 15 02:56:00 PM,7647511654,2016 Mar 15 02:21:00 PM,B,ANY,ALL OTHER,-,93XX BLOCK OF AURORA AVE N,47.696722,-122.344595,N3,North,N,-,704,ALL OTHER,All Other Offenses,90Z,-
1,2020-152553,2020 May 08 07:26:09 PM,13117279109,2020 May 08 11:40:00 AM,A,PROPERTY,BURGLARY,-,58XX BLOCK OF 5TH AVE NE,47.67117473,-122.32282696274,B3,North,B,WALLINGFORD,1545,PROPERTY CRIME,Burglary/Breaking & Entering,220,4500.2007
2,2026-903462,2026 Feb 22 09:01:19 AM,68657554298,2026 Feb 19 01:00:00 PM,A,PROPERTY,LARCENY-THEFT,-,92XX BLOCK OF 35TH AVE SW,47.52017448,-122.376792267112,F2,Southwest,F,ROXHILL/WESTWOOD/ARBOR HEIGHTS,4930,PROPERTY CRIME,All Other Larceny,23H,11402.3008
3,2015-361048,2015 Oct 15 02:05:00 PM,7693699654,2015 Oct 15 02:05:00 PM,A,PROPERTY,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,22XX BLOCK OF E MADISON ST,47.61879546,-122.303098812171,C2,East,C,-,5459,ALL OTHER,False Pretenses/Swindle/Confidence Game,26A,-
4,2016-437709,2016 Dec 05 07:19:00 PM,7695857797,2016 Dec 02 10:00:00 AM,A,PROPERTY,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,23XX BLOCK OF FRANKLIN AVE E,47.64087835,-122.324662544481,D3,West,D,-,5107,ALL OTHER,Credit Card/Automated Teller Machine Fraud,26B,-


## Section 2 — Data Profile

This section inspects the structure of the dataset before analysis. I use `df.head()`, `df.info()`, `df.describe()`, and `df.isnull().sum()` to understand what the data looks like, what column types it contains, and whether missing values may affect the analysis.

In [18]:
df.head()

,Report Number,Report DateTime,Offense ID,Offense Date,NIBRS Group AB,NIBRS Crime Against Category,Offense Sub Category,Shooting Type Group,Block Address,Latitude,Longitude,Beat,Precinct,Sector,Neighborhood,Reporting Area,Offense Category,NIBRS Offense Code Description,NIBRS_offense_code,Census Block 2020
0,2016-091229,2016 Mar 15 02:56:00 PM,7647511654,2016 Mar 15 02:21:00 PM,B,ANY,ALL OTHER,-,93XX BLOCK OF AURORA AVE N,47.696722,-122.344595,N3,North,N,-,704,ALL OTHER,All Other Offenses,90Z,-
1,2020-152553,2020 May 08 07:26:09 PM,13117279109,2020 May 08 11:40:00 AM,A,PROPERTY,BURGLARY,-,58XX BLOCK OF 5TH AVE NE,47.67117473,-122.32282696274,B3,North,B,WALLINGFORD,1545,PROPERTY CRIME,Burglary/Breaking & Entering,220,4500.2007
2,2026-903462,2026 Feb 22 09:01:19 AM,68657554298,2026 Feb 19 01:00:00 PM,A,PROPERTY,LARCENY-THEFT,-,92XX BLOCK OF 35TH AVE SW,47.52017448,-122.376792267112,F2,Southwest,F,ROXHILL/WESTWOOD/ARBOR HEIGHTS,4930,PROPERTY CRIME,All Other Larceny,23H,11402.3008
3,2015-361048,2015 Oct 15 02:05:00 PM,7693699654,2015 Oct 15 02:05:00 PM,A,PROPERTY,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,22XX BLOCK OF E MADISON ST,47.61879546,-122.303098812171,C2,East,C,-,5459,ALL OTHER,False Pretenses/Swindle/Confidence Game,26A,-
4,2016-437709,2016 Dec 05 07:19:00 PM,7695857797,2016 Dec 02 10:00:00 AM,A,PROPERTY,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,23XX BLOCK OF FRANKLIN AVE E,47.64087835,-122.324662544481,D3,West,D,-,5107,ALL OTHER,Credit Card/Automated Teller Machine Fraud,26B,-


The first five rows show that each row represents one reported crime incident. The dataset includes date fields, offense category fields, and location fields, which are the main variables needed for this analysis.

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Report Number                   5000 non-null   object
 1   Report DateTime                 5000 non-null   object
 2   Offense ID                      5000 non-null   int64 
 3   Offense Date                    5000 non-null   object
 4   NIBRS Group AB                  5000 non-null   object
 5   NIBRS Crime Against Category    5000 non-null   object
 6   Offense Sub Category            5000 non-null   object
 7   Shooting Type Group             5000 non-null   object
 8   Block Address                   5000 non-null   object
 9   Latitude                        5000 non-null   object
 10  Longitude                       5000 non-null   object
 11  Beat                            5000 non-null   object
 12  Precinct                        5000 non-null   

The dataset contains a mix of text columns and date-related columns. The offense date fields need to be converted into datetime format before I can analyze patterns by hour, month, year, or day of week.

In [20]:
df.describe()

,Offense ID
count,5.000000e+03
mean,1.963059e+10
std,1.953255e+10
min,7.625475e+09
25%,7.659741e+09
50%,7.690771e+09
75%,2.745316e+10
max,6.996407e+10


The descriptive statistics summarize the numeric fields in the dataset. After creating new time-based columns such as Hour, Month, and Year, this output helps confirm that the derived values fall within expected ranges.

In [21]:
df.isnull().sum()

Report Number                     0
Report DateTime                   0
Offense ID                        0
Offense Date                      0
NIBRS Group AB                    0
NIBRS Crime Against Category      0
Offense Sub Category              0
Shooting Type Group               0
Block Address                     0
Latitude                          0
Longitude                         0
Beat                              0
Precinct                          0
Sector                            0
Neighborhood                      0
Reporting Area                    0
Offense Category                  0
NIBRS Offense Code Description    0
NIBRS_offense_code                0
Census Block 2020                 0
dtype: int64

The missing value check shows which columns have incomplete data. Missing values matter most when they appear in the offense date, neighborhood, or offense sub-category fields because those fields are used directly in the visualizations.

## Additional Data Checks

Before cleaning the data, I check the exact column names and preview the fields that are most relevant to my questions: time, offense category, and location.

In [22]:
df.columns

Index(['Report Number', 'Report DateTime', 'Offense ID', 'Offense Date',
       'NIBRS Group AB', 'NIBRS Crime Against Category',
       'Offense Sub Category', 'Shooting Type Group', 'Block Address',
       'Latitude', 'Longitude', 'Beat', 'Precinct', 'Sector', 'Neighborhood',
       'Reporting Area', 'Offense Category', 'NIBRS Offense Code Description',
       'NIBRS_offense_code', 'Census Block 2020'],
      dtype='object')

In [23]:
df[[
    "Offense Date",
    "Report DateTime",
    "Offense Category",
    "Offense Sub Category",
    "Neighborhood",
    "Precinct",
    "Sector",
    "Beat"
]].head()

,Offense Date,Report DateTime,Offense Category,Offense Sub Category,Neighborhood,Precinct,Sector,Beat
0,2016 Mar 15 02:21:00 PM,2016 Mar 15 02:56:00 PM,ALL OTHER,ALL OTHER,-,North,N,N3
1,2020 May 08 11:40:00 AM,2020 May 08 07:26:09 PM,PROPERTY CRIME,BURGLARY,WALLINGFORD,North,B,B3
2,2026 Feb 19 01:00:00 PM,2026 Feb 22 09:01:19 AM,PROPERTY CRIME,LARCENY-THEFT,ROXHILL/WESTWOOD/ARBOR HEIGHTS,Southwest,F,F2
3,2015 Oct 15 02:05:00 PM,2015 Oct 15 02:05:00 PM,ALL OTHER,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,East,C,C2
4,2016 Dec 02 10:00:00 AM,2016 Dec 05 07:19:00 PM,ALL OTHER,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,West,D,D3


In [24]:
df["Offense Category"].value_counts().head(10)

Offense Category
PROPERTY CRIME    2524
ALL OTHER         2251
VIOLENT CRIME      225
Name: count, dtype: int64

These additional checks help me confirm that the dataset contains the fields needed for my three analysis questions. The offense category fields support offense-type analysis, while the neighborhood and date fields support spatial and temporal analysis.

## Data Cleaning and Preparation

To prepare the data for analysis, I convert the offense date and report date fields into datetime values. I then create new time-based columns for hour, month, year, and day of week. Rows without an offense date are removed because they cannot be used in the time-based charts.

In [25]:
# Convert the offense date and report date fields into datetime values.
df["Offense Date"] = pd.to_datetime(
    df["Offense Date"],
    format="mixed",
    errors="coerce"
)

df["Report DateTime"] = pd.to_datetime(
    df["Report DateTime"],
    format="mixed",
    errors="coerce"
)

# Rows without an offense date cannot be used for time-based analysis.
df = df.dropna(subset=["Offense Date"])

# Create time columns for analysis.
df["Hour"] = df["Offense Date"].dt.hour
df["Month"] = df["Offense Date"].dt.month
df["Year"] = df["Offense Date"].dt.year
df["Day of Week"] = df["Offense Date"].dt.day_name()

df[["Offense Date", "Hour", "Day of Week", "Month", "Year"]].head()

,Offense Date,Hour,Day of Week,Month,Year
0,2016-03-15 14:21:00,14,Tuesday,3,2016
1,2020-05-08 11:40:00,11,Friday,5,2020
2,2026-02-19 13:00:00,13,Thursday,2,2026
3,2015-10-15 14:05:00,14,Thursday,10,2015
4,2016-12-02 10:00:00,10,Friday,12,2016


The new time-based columns make it possible to analyze reported offenses by hour of day and to create broader time-period categories such as morning, afternoon, evening, and night.

## Section 3 — Analysis

This section includes three charts. Each chart answers one of the analytical questions from the Overview section and is followed by a written interpretation in plain language.

### Question 1: How do reported crime counts vary by hour of day in Seattle?

This chart compares the number of reported offenses across the 24 hours of the day.

In [26]:
# To answer the first question, I group the records by hour of day.
hourly_crimes = (
    df["Hour"]
    .value_counts()
    .sort_index()
    .reset_index()
)

# Rename columns so the chart code and labels are easier to understand.
hourly_crimes.columns = ["hour", "reported_offenses"]

# A bar chart works here because I am comparing counts across ordered hour categories.
fig1 = px.bar(
    hourly_crimes,
    x="hour",
    y="reported_offenses",
    title="Reported Crime Counts by Hour of Day in Seattle",
    labels={
        "hour": "Hour of Day",
        "reported_offenses": "Number of Reported Offenses"
    }
)

fig1.show()
fig1.write_image("charts/crime_counts_by_hour.png")

**Interpretation:**  
This chart shows how reported offenses are distributed across the 24 hours of the day. The pattern helps identify whether reports are concentrated around specific times rather than being evenly spread throughout the day. This answers the first question by showing that time of day is an important part of understanding reported crime patterns in Seattle.

### Question 2: Which Seattle neighborhoods have the highest number of reported offenses?

This chart compares the top 10 Seattle neighborhoods by number of reported offenses.

In [27]:
# To answer the second question, I count reported offenses by neighborhood.
# I remove "-" because it does not represent an actual neighborhood.
neighborhood_counts = (
    df[df["Neighborhood"] != "-"]["Neighborhood"]
    .dropna()
    .value_counts()
    .head(10)
    .reset_index()
)

# Rename columns for clearer chart labels.
neighborhood_counts.columns = ["neighborhood", "reported_offenses"]

# Reverse the order so the largest bar appears at the top in the horizontal chart.
neighborhood_counts = neighborhood_counts.sort_values(
    "reported_offenses",
    ascending=True
)

# A horizontal bar chart works well because neighborhood names can be long.
fig2 = px.bar(
    neighborhood_counts,
    x="reported_offenses",
    y="neighborhood",
    orientation="h",
    title="Top 10 Seattle Neighborhoods by Reported Offenses",
    labels={
        "reported_offenses": "Number of Reported Offenses",
        "neighborhood": "Neighborhood"
    }
)

fig2.show()
fig2.write_image("charts/top_10_neighborhoods_by_reported_offenses.png")

**Interpretation:**  
This chart shows that reported offenses are concentrated in a small number of Seattle neighborhoods. The top neighborhoods have noticeably higher report counts than the rest of the city. This should be interpreted carefully because higher report counts may reflect population density, commercial activity, policing patterns, or reporting behavior rather than only actual risk.

### Question 3: How do common offense sub-categories differ across morning, afternoon, evening, and night?

This chart groups reported offenses into four time periods and compares the most common offense sub-categories across those periods.

In [28]:
# To answer the third question, I group each record into a broad time period.
def assign_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["Time Period"] = df["Hour"].apply(assign_time_period)

# Remove vague categories so the chart focuses on meaningful offense types.
filtered_subcategories = df[
    ~df["Offense Sub Category"].isin(["ALL OTHER", "999"])
]

# Keep the four most common offense sub-categories so the grouped chart stays readable.
top_sub_categories = (
    filtered_subcategories["Offense Sub Category"]
    .dropna()
    .value_counts()
    .head(4)
    .index
)

# Count reports for each combination of time period and offense sub-category.
subcategory_time_counts = (
    filtered_subcategories[
        filtered_subcategories["Offense Sub Category"].isin(top_sub_categories)
    ]
    .groupby(["Time Period", "Offense Sub Category"])
    .size()
    .reset_index(name="reported_offenses")
)

# A grouped bar chart works here because I am comparing several offense sub-categories within each time period.
fig3 = px.bar(
    subcategory_time_counts,
    x="Time Period",
    y="reported_offenses",
    color="Offense Sub Category",
    barmode="group",
    title="Most Common Offense Sub-Categories by Time Period in Seattle",
    labels={
        "Time Period": "Time Period",
        "reported_offenses": "Number of Reported Offenses",
        "Offense Sub Category": "Offense Sub-Category"
    },
    category_orders={
        "Time Period": ["Morning", "Afternoon", "Evening", "Night"]
    }
)

fig3.show()
fig3.write_image("charts/offense_subcategories_by_time_period.png")

**Interpretation:**  
This chart compares common offense sub-categories across morning, afternoon, evening, and night. It shows whether the most frequent types of reported offenses follow similar daily patterns or become more common during specific time periods. This helps answer the third question by connecting offense type with time of day rather than treating all reported crimes as one category.

## Section 4 — Conclusions

### Question 1: How do reported crime counts vary by hour of day in Seattle?

The analysis shows that reported crime counts vary by hour rather than appearing evenly distributed across the day. This suggests that time of day is an important factor when interpreting reported crime patterns. If I had more time, I would compare weekday and weekend hourly patterns to see whether the timing changes across different routines.

### Question 2: Which Seattle neighborhoods have the highest number of reported offenses?

The neighborhood chart shows that reported offenses are concentrated in a limited number of neighborhoods. This suggests that place matters in the dataset, but the chart should be interpreted carefully because higher reports may reflect population density, commercial activity, nightlife, policing patterns, or reporting behavior rather than only actual risk. A next step would be to normalize the counts by population or visitor activity.

### Question 3: How do common offense sub-categories differ across time periods?

The offense sub-category chart shows that different offense types can have different patterns across morning, afternoon, evening, and night. This suggests that broad crime totals hide more specific differences in when certain types of incidents are reported. If I continued this project, I would compare more sub-categories and investigate whether these time patterns changed across years.

## Section 5 — Process

I began this project from my A6 visualization work, where I created the first set of charts for the Seattle crime dataset. For MP1b, I reorganized the notebook into a clearer data analysis story with an overview, data profile, analysis, conclusions, and process reflection.

One important part of the process was narrowing my questions so each chart answered one specific question. My first question originally included hour of day, day of week, and month, but the final chart focuses on hour of day because that made the analysis more direct and easier to interpret.

I used pandas to inspect the dataset, check column names, convert date fields into datetime values, and create new columns such as Hour, Month, Year, Day of Week, and Time Period. I also filtered out vague or missing categories such as "-" for neighborhoods and "ALL OTHER" or "999" for offense sub-categories when they made the charts harder to interpret.

I used AI tools to help me think through the structure of the notebook and debug parts of the code, but I did not treat the AI output as final. I checked whether the code ran from top to bottom, whether each chart matched the question it was supposed to answer, and whether the written interpretation was supported by the chart.

This process connected to the Week 7 discussion about calibrated skepticism. AI tools were useful for creating starting structures and helping with code, but I still needed to ask what evidence supported each finding, what might be missing from the dataset, and whether the final claims were careful enough to put my name on.